# Llama 2 7B Fine-tuning Environment Setup (Colab Pro+)

This notebook guides through the essential steps to set up a Google Colab Pro+ environment for fine-tuning the Llama 2 7B model. The setup targets an A100 GPU and a High-RAM runtime for optimal performance during fine-tuning.

## Section 1: Colab Pro+ Configuration

To effectively fine-tune larger models like Llama 2 7B, it's crucial to configure your Colab environment correctly:

1.  **Ensure you have Colab Pro or Pro+**: These versions provide access to premium GPUs and higher RAM configurations.
2.  **Select GPU**: 
    *   Navigate to `Runtime` -> `Change runtime type`.
    *   Under `Hardware accelerator`, select `GPU`.
    *   In the `GPU type` dropdown, choose `A100 GPU`. If A100 is not available, T4 or V100 might work for smaller experiments but A100 is recommended for 7B models.
3.  **Select RAM**: 
    *   In the same `Runtime type` settings, under `Runtime shape`, select `High-RAM`.
4.  **Save Settings**: Click `Save`.

## Section 2: Mount Google Drive

Mounting your Google Drive is highly recommended for several reasons:
-   **Persistent Storage**: Store your datasets, model checkpoints, and fine-tuning outputs.
-   **Easy Access**: Load data and save results without re-uploading each session.
-   **Continuity**: If your Colab runtime disconnects, your data remains safe in Drive.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# print("Google Drive mounted. You can now access your files via /content/drive/My Drive/")

## Section 3: Install Essential Libraries

The following libraries are crucial for fine-tuning Llama 2 with Hugging Face tools:

-   **`torch`, `torchvision`, `torchaudio`**: PyTorch and its ecosystem for tensor computations and neural network building, ensuring CUDA compatibility (cu118 for recent NVIDIA drivers).
-   **`transformers`**: Hugging Face's library for accessing pre-trained models (like Llama 2), tokenizers, and pipelines.
-   **`accelerate`**: Simplifies running PyTorch training scripts on various distributed configurations (multi-GPU, TPU) and with mixed precision.
-   **`bitsandbytes`**: Enables 4-bit quantization (like QLoRA), significantly reducing memory footprint for loading large models.
-   **`sentencepiece`**: Often used as a tokenizer for models like Llama.
-   **`trl`**: (Transformer Reinforcement Learning) Library from Hugging Face for fine-tuning language models using Reinforcement Learning (e.g., PPO) and Supervised Fine-tuning (SFT).
-   **`peft`**: (Parameter-Efficient Fine-Tuning) Library for applying methods like LoRA, QLoRA, which allow fine-tuning only a small subset of model parameters.
-   **`datasets`**: Hugging Face library for easily loading and processing datasets.
-   **`huggingface_hub`**: For interacting with the Hugging Face Hub (downloading models, pushing results).

In [ ]:
# Ensure the PyTorch version is compatible with your CUDA version (cu118 for Colab A100s typically)
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Install transformers and related libraries for fine-tuning
# !pip install transformers accelerate bitsandbytes sentencepiece

# Install TRL (Transformer Reinforcement Learning), PEFT (Parameter-Efficient Fine-Tuning), and Datasets
# !pip install trl peft datasets

# For Hugging Face Hub interactions
# !pip install huggingface_hub

## Section 4: Hugging Face Hub Authentication

To download the official Llama 2 models or to push your fine-tuned models to the Hugging Face Hub, you need to authenticate.

1.  **Get a Token**: If you don't have one, go to your Hugging Face profile -> `Settings` -> `Access Tokens` -> `New token`. Give it a name and `write` permissions if you plan to upload models.
2.  **Login**: Run the code cell below. It will prompt you to enter your token.

In [ ]:
# from huggingface_hub import login
# login()

# After running, paste your Hugging Face Hub token when prompted and press Enter.

## Section 5: Verify Setup (Optional)

It's a good practice to verify that the libraries are installed correctly and that PyTorch can detect the GPU.

In [ ]:
# import torch
# print(f"PyTorch version: {torch.__version__}")
# print(f"CUDA available: {torch.cuda.is_available()}")
# if torch.cuda.is_available():
#     print(f"GPU: {torch.cuda.get_device_name(0)}")
#     print(f"CUDA version by PyTorch: {torch.version.cuda}")
# else:
#     print("CUDA is not available. Check your runtime configuration and library installations.")

## Section 6: Load Llama 2 7B Model and Tokenizer

This section outlines how to load the Llama 2 7B model and its tokenizer using the Hugging Face `transformers` library. We'll use the model identifier `meta-llama/Llama-2-7b-hf`. Access to Llama 2 models requires approval from Meta and authentication via Hugging Face Hub (completed in Section 4).

To manage memory effectively, especially on single GPU setups, we'll load the model with 4-bit quantization using `bitsandbytes`. This significantly reduces the model's memory footprint.

In [ ]:
# from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# import torch

# model_name = "meta-llama/Llama-2-7b-hf" # Or other Llama 2 variant

# # Configuration for 4-bit quantization
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16, # or torch.bfloat16 for A100
#     bnb_4bit_use_double_quant=False,
# )

# # Load model
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     quantization_config=bnb_config,
#     device_map="auto", # Automatically distribute across available GPUs
#     # use_auth_token=True # Not strictly needed if login() was successful
# )
# print("Model loaded.")

# # Load tokenizer
# tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token # Set pad token to EOS token for open-ended generation
# tokenizer.padding_side = "right" # Fix for weird overflow issue with fp16 training
# print("Tokenizer loaded.")